# Dependency Caching with uv — Bob Follows the Cache

*A small project, two kinds of wheel, and a cache Bob can actually inspect.*

## Introduction — Bob’s Cache Question

> **Bob:** A fresh virtual environment is cheap if I can reuse yesterday's downloads. But what does uv keep, and what exactly do its cleanup commands throw away?

Run these Bash cells from top to bottom in this lab's Dev Container. The full explanation and matching commands are in [Dependency Caching with uv](../../docs/chapter-03/section-03.md). The image supplies uv and Python; *you* populate the package cache. The main `.venv` is installed with `UV_LINK_MODE=copy` so imports survive cache cleanup. Cache sizes and internal directory names can vary across uv versions.

## Set Up the Lab

> **Bob:** I will keep my experiments in this folder and leave the host's uv cache alone.

The first cell clears only this lab's generated files. Restart the container for an empty container-local uv cache; do not clear a cache shared with unrelated work.

In [ ]:
%%bash
set -euxo pipefail
pwd
uv --version
uv cache dir
rm -rf lab .venv reports src/uv_cache_demo/data/embeddings.bin
mkdir -p lab


## Populate and Inspect the Cache

### Downloaded Wheels and a Built Local Wheel

> **Bob:** NumPy arrives as a downloaded wheel. My own package must be built from source. I want both in the cache before I compare cleanup commands.

The generator creates a deterministic 16 MiB fixture. `--no-editable` builds a regular project wheel; `CACHE_DEMO_KEY` fingerprints its sources and fixture so the same wheel can be reused on a later run. Inspect the cache and the installed program:

In [ ]:
%%bash
set -euxo pipefail
uv sync --locked --no-install-project --python 3.12
.venv/bin/python scripts/generate_fixture.py
CACHE_DEMO_KEY=$(.venv/bin/python scripts/build_key.py) uv sync --locked --no-editable --python 3.12
.venv/bin/python -m uv_cache_demo.main --output reports/embeddings.png
uv cache size --human --preview-features cache-size
find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print


### An Unzipped Source Distribution

> **Bob:** A source distribution has to be unpacked and built. Can I see that extra cache material before CI discards it?

Force a source build of the small `idna` package into a disposable target. The package is independent of the locked demo; this adds source/build data to uv's cache without changing `uv.lock`. `sdists-v*` buckets are implementation details, so examine what your uv version creates:

In [ ]:
%%bash
set -euxo pipefail
uv pip install --python "$(uv python find 3.12)" --target lab/sdist-target --no-deps --no-binary idna 'idna==3.10'
find "$(uv cache dir)" -type d -path '*/sdists-v*/pypi/idna/3.10/*/src/idna' -print
find "$(uv cache dir)" -type f -name 'idna-*.whl' -print
uv cache size --human --preview-features cache-size


## Reuse the Cache

### Reinstall Offline

> **Bob:** I can delete an environment without deleting downloaded packages. Let me rebuild it with the network switched off.

The locked dependency wheels are cached; the fixture is still present. The fingerprint also lets uv reuse the locally built wheel:

In [ ]:
%%bash
set -euxo pipefail
rm -rf .venv
CACHE_DEMO_KEY=$("$(uv python find 3.12)" scripts/build_key.py) uv sync --locked --no-editable --offline --python 3.12
.venv/bin/python -c 'import numpy, uv_cache_demo; print(numpy.__file__, uv_cache_demo.__file__)'


## Compare UV_LINK_MODE

These are *installation* modes, not four different caches. Each trial installs cached NumPy into its own disposable environment; `--offline` proves it uses cached files. `stat` reports the installed file type and number of hard links. Keep the symlink trial separate: deleting its cache would break it.

### Clone — The Linux Default

> **Bob:** Copy-on-write cloning can share disk blocks until I modify a file, when the filesystem supports it. The lab overrides Linux's normal `clone` default with `copy`, so I'll opt in explicitly here.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/clone UV_LINK_MODE=clone uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/clone/lib/python3.12/site-packages/numpy/__init__.py
lab/clone/bin/python -c 'import numpy; print(numpy.__version__)'


### Copy — The Lab’s Default

> **Bob:** Independent copies cost space, but cleaning the cache cannot remove their contents. That's why my main `.venv` uses `copy`.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/copy UV_LINK_MODE=copy uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/copy/lib/python3.12/site-packages/numpy/__init__.py


### Hardlink — Share Files on One Filesystem

> **Bob:** Hard links point to the same file contents. They need the cache and environment on the same filesystem; editing one shared file may affect another installation.

Here the container-local cache and workspace must share a filesystem for hardlinking to succeed. uv warns and falls back to copying when they do not; check the reported link count rather than assuming it worked.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/hardlink UV_LINK_MODE=hardlink uv sync --locked --no-install-project --offline --python 3.12
stat -c '%F; hard links: %h' lab/hardlink/lib/python3.12/site-packages/numpy/__init__.py


### Symlink — Depend on the Cache

> **Bob:** A symlink resolves into the cache. If I clean that cache, this installation will break. I'll remove this trial *before* I prune.

Resolve the target with `readlink`, then discard this disposable environment; the main copy-mode environment stays available for the cleanup demonstrations.

In [ ]:
%%bash
set -euxo pipefail
UV_PROJECT_ENVIRONMENT=lab/symlink UV_LINK_MODE=symlink uv sync --locked --no-install-project --offline --python 3.12
readlink lab/symlink/lib/python3.12/site-packages/numpy/__init__.py
rm -rf lab/symlink


## Clean the Cache in Stages

These are successive experiments, not a ranking by how little they remove: `--ci` can remove *more* than ordinary prune. The notebook adds a new target between them so each action has something visible to remove. Cache sizes vary; compare before and after rather than relying on fixed numbers.

### CI Prune — Remove Downloads and Unzipped Sources

> **Bob:** A CI runner can download ready-made wheels again, but I would rather keep a wheel I built myself. Does the installed project still work?

`uv cache prune --ci` removes pre-built downloads and unzipped source distributions, retains wheels built from source, and does not delete `.venv`. Check the same local wheel path before and after:

In [ ]:
%%bash
set -euxo pipefail
find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print
find "$(uv cache dir)" -type d -path '*/sdists-v*/pypi/idna/3.10/*/src/idna' -print
uv cache size --human --preview-features cache-size
uv cache prune --ci
find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print
find "$(uv cache dir)" -type f -name 'idna-*.whl' -print
find "$(uv cache dir)" -type d -path '*/sdists-v*/pypi/idna/3.10/*/src/idna' -print
uv cache size --human --preview-features cache-size
.venv/bin/python -c 'import numpy; from uv_cache_demo.main import cache_message; print(cache_message())'


### Normal Prune — Remove a Centralized Environment

> **Bob:** Normally `.venv` sits beside `pyproject.toml`. Can I deliberately put a separate project environment in the cache, then see what ordinary pruning does to it?

Copy the locked declarations to an ignored disposable project. `centralized-project-envs` is opt-in preview behavior: uv stores its default environment in the cache and attempts to link `.venv` to it. Do not set `UV_PROJECT_ENVIRONMENT` or use `--active`; those select explicit paths and bypass centralization. Save the resolved target before pruning because `.venv` may become a broken link afterward:

In [ ]:
%%bash
set -euxo pipefail
mkdir -p lab/centralized
cp pyproject.toml uv.lock lab/centralized/
uv --preview-features centralized-project-envs sync --directory lab/centralized --locked --no-install-project --python 3.12
ls -ld lab/centralized/.venv
readlink -f lab/centralized/.venv > lab/centralized-target
test -d "$(cat lab/centralized-target)"
lab/centralized/.venv/bin/python -c 'import numpy; print(numpy.__version__)'


> **Bob:** Ordinary pruning removes unused entries and centralized environments. My *main* `.venv` is still a regular copy-mode environment outside the cache.

Check the centralized target and the remaining built wheel after pruning. The ordinary prune need not reduce the cache by more than CI pruning did; they have different targets:

In [ ]:
%%bash
set -euxo pipefail
uv cache prune
test ! -d "$(cat lab/centralized-target)"
find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print
.venv/bin/python -c 'import numpy; print(numpy.__version__)'
uv cache size --human --preview-features cache-size


### Clean — Remove All Cache Entries

> **Bob:** `uv cache clean` leaves no reusable wheels. Does it remove a package I already copied into my environment?

The cache is emptied; the main copy-mode `.venv` and the report are separate files. Check both after cleaning:

In [ ]:
%%bash
set -euxo pipefail
uv cache clean
uv cache size --human --preview-features cache-size
test -z "$(find "$(uv cache dir)" -type f -name 'uv_cache_demo-*.whl' -print)"
.venv/bin/python -c 'import numpy; from uv_cache_demo.main import cache_message; print(cache_message())'
test -f reports/embeddings.png


## Epilogue — Bob’s Runbook

> **Bob:** I can delete `.venv` and reinstall offline while the cache is full. CI pruning keeps source-built wheels; ordinary pruning removes unused data and centralized environments; cleaning clears it all. With `copy`, my installed packages survive. With `symlink`, they would depend on the cache staying put.

For the runner-side version, compare the [GitHub Actions workflow](../../.github/workflows/uv-cache.yml) and [its explanation](../../docs/chapter-03/section-03.md#using-uvs-cache-in-ci-environments).